# Data Preprocessing 5 - Text Data with NLTK

> **MLCourse · Data Science Foundations · 05_data_preprocessing**

The pure-Python pipeline showed you the mechanics. NLTK gives you TRAINED
tokenizers, curated corpora, mature stemmers/lemmatizers and sentiment - the
toolkit real projects start from.

### What you'll learn
- Downloading & guarding NLTK resources
- Sentence + word tokenization (3 tokenizers compared)
- The stopwords corpus, multilingual peek, domain extension
- Porter vs Lancaster vs Snowball stemmers side-by-side
- WordNet lemmatization and the critical `pos=` lesson
- POS tagging → smarter lemmatization via a tag mapper
- A complete reusable NLTK preprocess function
- VADER lexicon sentiment scoring
- When to choose stdlib vs NLTK

In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import string

np.random.seed(42)

import nltk

def ensure_nltk(pkg: str, path: str) -> None:
    """Download an NLTK resource once; degrade gracefully when offline."""
    try:
        nltk.data.find(path)
    except LookupError:
        try:
            nltk.download(pkg)
        except Exception as e:
            print(f"offline? skipping {pkg}: {e}")

for pkg, path in [
    ("punkt", "tokenizers/punkt"),           # sentence splitter model
    ("punkt_tab", "tokenizers/punkt_tab"),   # newer punkt format (nltk>=3.8.2)
    ("stopwords", "corpora/stopwords"),
    ("wordnet", "corpora/wordnet"),
    ("omw-1.4", "corpora/omw-1.4"),          # wordnet dependency
]:
    ensure_nltk(pkg, path)

print("NLTK setup attempted.")

NLTK setup attempted.


[nltk_data] Downloading package wordnet to C:\Users\Thoyajaksha
[nltk_data]     Kashyap\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Thoyajaksha
[nltk_data]     Kashyap\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


### 1. Tokenization - trained splitters

`.split()` broke on "don't" and "U.S.A.". Let's see what proper models do.

In [2]:
from nltk.tokenize import sent_tokenize, word_tokenize, wordpunct_tokenize

paragraph = (
    "Dr. Smith didn't trust the 3rd-party report. It cost $1,500! "
    "Well... he was right, wasn't he?"
)

print("SENTENCES:")
for s in sent_tokenize(paragraph):
    print("  •", s)

print("\nWORD TOKENIZERS on one tricky string:")
tricky = "Don't split U.S.A. badly, please."
compare = pd.DataFrame({
    "tokenizer": ["word_tokenize\n(Treebank)", "wordpunct_tokenize"],
    "output": [str(word_tokenize(tricky)), str(wordpunct_tokenize(tricky))],
})
compare.style.hide(axis="index")

SENTENCES:
  • Dr. Smith didn't trust the 3rd-party report.
  • It cost $1,500!
  • Well... he was right, wasn't he?

WORD TOKENIZERS on one tricky string:


tokenizer,output
word_tokenize (Treebank),"['Do', ""n't"", 'split', 'U.S.A.', 'badly', ',', 'please', '.']"
wordpunct_tokenize,"['Don', ""'"", 't', 'split', 'U', '.', 'S', '.', 'A', '.', 'badly', ',', 'please', '.']"


Note Treebank quirks: "Don't" → `["Do", "n't"]` (contractions split), periods
kept as tokens. `wordpunct` is cruder: splits at every punctuation char.
Choose per downstream need - most pipelines use `word_tokenize`.

### 2. Stopwords - a real corpus instead of our hand list


In [3]:
from nltk.corpus import stopwords

eng_sw = set(stopwords.words("english"))          # SET for O(1) membership!
print(f"english stopwords: {len(eng_sw)} words")
print(sorted(list(eng_sw))[:15], "...")

print("\nmultilingual peek:", stopwords.words("spanish")[:6])

# Extend with what hurts OUR task:
domain_stop = eng_sw | {"movie", "film"}          # e.g., reviews corpus
punct_set = set(string.punctuation)
full_stop = domain_stop | punct_set

tokens = word_tokenize("The movie was not good , it was terrible !")
kept = [w for w in tokens if w.lower() not in full_stop]
print("\ntokens :", tokens)
print("filtered:", kept)
# ⚠️ Notice we just deleted 'not' - stopword removal can DESTROY sentiment signal!

english stopwords: 198 words
['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't"] ...

multilingual peek: ['de', 'la', 'que', 'el', 'en', 'y']

tokens : ['The', 'movie', 'was', 'not', 'good', ',', 'it', 'was', 'terrible', '!']
filtered: ['good', 'terrible']


### 3. Stemming - three aggressiveness levels


In [4]:
from nltk.stem import PorterStemmer, LancasterStemmer, SnowballStemmer

porter   = PorterStemmer()                        # classic, moderate
lancaster= LancasterStemmer()                     # the aggressive one
snowball = SnowballStemmer("english")             # improved Porter ("porter2")

test_words = ["studies", "studying", "running", "better",
              "universities", "happily", "flies", "generalization"]

stem_df = pd.DataFrame({
    "word": test_words,
    "Porter": [porter.stem(w) for w in test_words],
    "Lancaster": [lancaster.stem(w) for w in test_words],
    "Snowball": [snowball.stem(w) for w in test_words],
})
stem_df

,word,Porter,Lancaster,Snowball
0,studies,studi,study,studi
1,studying,studi,study,studi
2,running,run,run,run
3,better,better,bet,better
4,universities,univers,univers,univers
5,happily,happili,happy,happili
6,flies,fli,fli,fli
7,generalization,gener,gen,general


Observations: all three butcher irregulars (`better`, `flies`) because they only
strip suffixes by rule - no dictionary. Lancaster sometimes over-trims into
non-words (`happily→happy` ok but `generalization→gen`!). **Snowball** is today's
sane default when you must stem.

### 4. Lemmatization with WordNet - the POS lesson

A lemma is a REAL dictionary word. But WordNet needs to know the part of speech,
or it assumes NOUN.

In [5]:
from nltk.stem import WordNetLemmatizer

wnl = WordNetLemmatizer()

probe = [("studies", "n"), ("studies", "v"),
         ("running", "n"), ("running", "v"),
         ("better", "a"),  ("went", "v")]

pd.DataFrame({
    "word": [w for w, _ in probe],
    "pos": [p for _, p in probe],
    "lemma": [wnl.lemmatize(w, pos=p) for w, p in probe],
})

,word,pos,lemma
0,studies,n,study
1,studies,v,study
2,running,n,running
3,running,v,run
4,better,a,good
5,went,v,go


Same input, different answers depending on `pos`: *studies* as noun stays
"studies" (plural of study-noun → actually needs 'n' handling of 'ies'), as verb
becomes "study". `better` only collapses to `good` as ADJECTIVE.

> ⚠️ **Common pitfall:** default `pos='n'` makes lemmatizers look useless on
verbs. Tag first, then lemmatize.

### 5. POS tagging → automatic smarter lemmas


In [6]:
from nltk import pos_tag

sentence = word_tokenize("The quick foxes were jumping over lazy dogs yesterday")
tagged = pos_tag(sentence)
print(tagged[:6])

# Treebank tags → WordNet POS mapping:
def wordnet_pos(treebank_tag: str) -> str:
    """Map Penn-Treebank tag prefix -> WordNet pos letter."""
    if treebank_tag.startswith("J"):
        return "a"            # adjective
    if treebank_tag.startswith("V"):
        return "v"            # verb
    if treebank_tag.startswith("R"):
        return "r"            # adverb
    return "n"                # default noun

smart_lemmas = [wnl.lemmatize(w, pos=wordnet_pos(t)) for w, t in tagged]
pd.DataFrame({"token": sentence, "tag": [t for _, t in tagged],
              "lemma(smart)": smart_lemmas}).head(8)

[('The', 'DT'), ('quick', 'JJ'), ('foxes', 'NNS'), ('were', 'VBD'), ('jumping', 'VBG'), ('over', 'IN')]


,token,tag,lemma(smart)
0,The,DT,The
1,quick,JJ,quick
2,foxes,NNS,fox
3,were,VBD,be
4,jumping,VBG,jump
5,over,IN,over
6,lazy,JJ,lazy
7,dogs,NNS,dog


"foxes→fox", "were→be", "jumping→jump", "dogs→dog" - all correct automatically.

### 6. The complete NLTK preprocessing function


In [7]:
def preprocess_nltk(text: str, keep_negations: bool = False):
    """
    Full pipeline: sentences -> tokens -> clean -> POS-aware lemmatize.
    Returns list of lemma strings.
    """
    tokens = word_tokenize(text.lower())                       # 1. tokenize lowercase
    if keep_negations:
        sw = {w for w in (stopwords.words("english")) if w != "not"}
    else:
        sw = set(stopwords.words("english"))
    sw |= set(string.punctuation)                              # 2. stoplist + punct
    kept = [t for t in tokens if t.isalpha()]                  # 3. drop numbers/junk
    kept = [t for t in kept if t not in sw]                    # 4. stopword filter
    tagged = pos_tag(kept)                                     # 5. POS tags
    return [wnl.lemmatize(w, pos=wordnet_pos(t)) for w, t in tagged]  # 6. lemmatize

corpus = [
    "<p>The movies weren't great , honestly .</p>",
    "I LOVE this product!!! Best purchase ever :)",
    "Terrible support ; waited 3 weeks for refund.",
    "It's okay I guess ... nothing special about it.",
]

rows = [{"review": c,
         "tokens": preprocess_nltk(c),
         "negation_kept": preprocess_nltk(c, keep_negations=True)}
        for c in corpus]
pd.DataFrame(rows)

,review,tokens,negation_kept
0,"<p>The movies weren't great , honestly .</p>","[p, movie, great, honestly]","[p, movie, great, honestly]"
1,I LOVE this product!!! Best purchase ever :),"[love, product, best, purchase, ever]","[love, product, best, purchase, ever]"
2,Terrible support ; waited 3 weeks for refund.,"[terrible, support, wait, week, refund]","[terrible, support, wait, week, refund]"
3,It's okay I guess ... nothing special about it.,"[okay, guess, nothing, special]","[okay, guess, nothing, special]"


### Compare `weren't` rows: keeping negation preserves the complaint's polarity -
### a one-flag difference that flips sentiment systems.


### 7. VADER - lexicon sentiment out of the box

VADER (Valence Aware Dictionary and sEntiment Reasoner) scores text using a
crowdsourced word/rule lexicon: negations, intensifiers, punctuation.

In [8]:
from nltk.sentiment import SentimentIntensityAnalyzer

try:
    ensure_nltk("vader_lexicon", "sentiment/vader_lexicon")
    sia = SentimentIntensityAnalyzer()

    probes = [
        "I absolutely LOVED it!!",                 # positive + caps +
        "This was not good.",                      # negation flips valence
        "meh, it was fine I suppose...",           # weak/mixed
        "utter waste of money :(",                 # negative + emoticon
    ]
    vader_df = pd.DataFrame(
        [{"text": p, **sia.polarity_scores(p)} for p in probes]
    )
    vader_df
except Exception as e:
    print("VADER unavailable offline:", e)

[nltk_data] Downloading package vader_lexicon to C:\Users\Thoyajaksha
[nltk_data]     Kashyap\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


Reading `compound`: normalized score -1..+1; common thresholds
`≥ +0.05 → positive`, `≤ -0.05 → negative`, else neutral. Sarcasm and long
documents are its known blind spots - honest limitations to state up front.

### 8. stdlib vs NLTK - same review, two pipelines


In [9]:
import re

def stdlib_tokens(text: str):
    """Our transparent pipeline from notebook 04."""
    text = re.sub(r"<[^>]+>", " ", text.lower())
    text = re.sub(r"[^a-z\s]", " ", text)          # letters+spaces only
    hand_sw = {"the","was","it","i","this","and","of","to","a","in","my"}
    return [w for w in text.split() if w not in hand_sw]

sample = corpus[0]
left, right = stdlib_tokens(sample), preprocess_nltk(sample)

pd.DataFrame({
    "stdlib pipeline": pd.Series(left),
    "nltk pipeline": pd.Series(right),
})

,stdlib pipeline,nltk pipeline
0,movies,p
1,weren,movie
2,t,great
3,great,honestly
4,honestly,NaN


| Criterion | Pure stdlib | NLTK |
### |---|---|---|
### | dependencies | none | pip install + ~30 MB corpora |
### | transparency | total (you wrote it) | library internals |
### | tokenizer quality | regex approximations | trained Treebank rules |
### | morphology | naive rules | WordNet lemmas + POS |
### | extras | DIY everything | sentiment/tagging/corpora built-in |

Rule of thumb: **learn concepts stdlib-first** (this course's order!), then use
NLTK (or spaCy, the faster production successor) for real work.

### Summary & key takeaways

- `sent_tokenize` → `word_tokenize`; Treebank splits contractions deliberately.
- Real stopword lists live in corpora; extend with domain terms; beware deleting
  **negations** like "not".
- Stemmers: Porter < Snowball < Lancaster in aggression; all fail on irregulars;
  lemmatizers output true dictionary words but NEED `pos=`.
- Map Treebank tags → WordNet POS, then lemmatize - the professional recipe.
- Wrap downloads in guarded helpers so offline runs degrade gracefully.
- VADER gives instant `compound` sentiment (-1..1) with honest limitations.
- spaCy is the production next step after these fundamentals.